# False verification reporting: DoM and CC++ comparison

Separate, private representation pilot. No response generation, live hacking, old-study edits or fine-tuning. The fixtures test consistency with an explicit verification record, not dishonest intent. Run in `frederic.reichelt@gmail.com`, on T4 for capture; CPU for the remaining steps.

Upload the frozen kit and `scripts/audit_false_reporting_text_vocab_v1.py` through the Files panel before running. The separate adapter preserves the original TF-IDF artifacts and never loads object pickles. See `docs/deception_monitoring/false_reporting_dom_v1/AUDIT-REPAIR.md`.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from google.colab import files, drive, userdata

EXPECTED_ACCOUNT = 'frederic.reichelt@gmail.com'
EXPECTED_SHA256 = '373474f0112b31f55659f70d194abe9c147bc6bea322964820750a92b6ac1b5b'
local_kit = Path('/content/false-reporting-dom-v1-private-kit.zip')
uploaded = {local_kit.name: local_kit.read_bytes()} if local_kit.exists() else files.upload()
assert len(uploaded) == 1, 'Upload only the frozen private code kit.'
filename, payload = next(iter(uploaded.items()))
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256, 'Code kit checksum differs.'
SOURCE = Path('/content/false-reporting-code-' + EXPECTED_SHA256[:12])
SOURCE.mkdir(exist_ok=True)
with zipfile.ZipFile(__import__('io').BytesIO(payload)) as archive:
    for item in archive.infolist():
        path = Path(item.filename)
        assert not path.is_absolute() and '..' not in path.parts, 'Unsafe archive path.'
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink not allowed.'
        target = SOURCE / path
        target.parent.mkdir(parents=True, exist_ok=True)
        content = archive.read(item)
        if target.exists():
            assert target.read_bytes() == content, 'Existing source differs.'
        else:
            target.write_bytes(content)
del uploaded, payload
design = json.loads((SOURCE / 'prepared/design.json').read_text())
for name, expected in design['source_sha256'].items():
    assert hashlib.sha256((SOURCE / name).read_bytes()).hexdigest() == expected
print('Frozen private kit verified; account expected:', EXPECTED_ACCOUNT)


## Durable storage before inference

Use the approved account shown in Colab. A disconnect preserves completed captures and receipts on Drive. This namespace does not replace the old deception study.


In [ ]:
drive.mount('/content/drive')
design_bytes = (SOURCE / 'prepared/design.json').read_bytes()
design_sha = hashlib.sha256(design_bytes).hexdigest()
OUTPUT = Path('/content/drive/MyDrive/verified-reasoning-monitoring/reward-hacking/false-reporting-dom-v1-' + design_sha[:12])
OUTPUT.mkdir(parents=True, exist_ok=True)
for name in ('design.json', 'manifest.json'):
    target = OUTPUT / name
    content = (SOURCE / 'prepared' / name).read_bytes()
    if target.exists():
        assert target.read_bytes() == content, 'Durable prepared input differs.'
    else:
        target.write_bytes(content)
print('Prepared inputs saved:', OUTPUT)


## Pinned packages and fresh-process runner

Secrets are read inside the Colab UI and passed privately only to the capture child process. They are not printed, written into result files or put on a command line.


In [ ]:
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE / 'requirements-false-reporting.txt')],
    cwd=SOURCE, text=True, capture_output=True)
(OUTPUT / 'package-install.log').write_text(installation.stdout + installation.stderr)
assert installation.returncode == 0, 'Package installation failed; inspect the private log.'
ENV = dict(os.environ, OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
           MKL_NUM_THREADS='2', PYTHONDONTWRITEBYTECODE='1',
           PYTHONPATH=str(SOURCE / 'src') + ':' + str(SOURCE / 'scripts'))
def execute(arguments, name, capture_secret=False):
    env = dict(ENV)
    secret = userdata.get('HF_TOKEN') if capture_secret else None
    if secret:
        env['HF_TOKEN'] = secret
    try:
        with (OUTPUT / (name + '.log')).open('a') as log:
            process = subprocess.Popen([sys.executable] + [str(value) for value in arguments],
                                       cwd=SOURCE, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            try:
                for line in process.stdout:
                    safe = line.replace(secret, '[REDACTED]') if secret else line
                    log.write(safe)
                    log.flush()
                    print(safe.rstrip(), flush=True)
                code = process.wait()
            finally:
                if process.poll() is None:
                    process.terminate()
                    process.wait()
        if code:
            raise RuntimeError(name + ' failed; preserve files and inspect its log.')
    finally:
        env.pop('HF_TOKEN', None)
        secret = None
def raw(mode, *extra):
    execute([SOURCE / 'scripts/run_false_reporting_dom_v1.py', mode,
             '--source', SOURCE, '--output', OUTPUT, *extra],
            'raw-' + mode, capture_secret=(mode == 'capture'))
print('Fresh CLI execution ready; no model loaded in the notebook kernel.')


## T4 capture, intentional interruption, resume

Every final-report token is captured at all 28 registered layer indices. The first capture compares dense and chunked extraction. An intentional seven-row interruption is followed by checksum-verified resume. Operational 15-minute pauses preserve progress and are not scientific failures.


In [ ]:
if not (OUTPUT / 'interruption.json').exists():
    raw('capture', '--stop-after', '7')
assert (OUTPUT / 'interruption.json').exists(), 'Interruption receipt missing.'


In [ ]:
while not (OUTPUT / 'capture-complete.json').exists():
    before = len(list((OUTPUT / 'captures').glob('*.json')))
    raw('capture')
    after = len(list((OUTPUT / 'captures').glob('*.json')))
    if after <= before and not (OUTPUT / 'capture-complete.json').exists():
        raise RuntimeError('No new capture progress; preserve receipts and diagnose before retry.')
print('All 456 captures complete; model process released.')


## First comparison: raw DoM and baselines

Fit only on training, choose each layer and cutoff only on validation, then evaluate both untouched held-out splits. No scores are used to revise the fixtures.


In [ ]:
if not (OUTPUT / 'results.json').exists():
    raw('analyze')
adapter = Path('/content/audit_false_reporting_text_vocab_v1.py')
assert hashlib.sha256(adapter.read_bytes()).hexdigest() == 'a99c87384b702811eb6cc416d9d71e3f32f29568fa221fc2528864368bbedf3b'
execute([adapter, '--source', SOURCE, '--output', OUTPUT], 'raw-safe-vocabulary-audit')
shutil.copyfile(adapter, OUTPUT / adapter.name)
assert json.loads((OUTPUT / 'audit.json').read_text())['status'] == 'verified_captures_directions_selection_and_all_comparator_projections'


## Second comparison: one component at a time

Four learned heads separate single/all-layer features, sliding-window smoothing, and softmax loss weighting. Each has three fixed readouts. All 12 arms are reported, not a test-selected winner. CPU fitting uses two threads and one capture at a time.


In [ ]:
execute([SOURCE / 'scripts/run_false_reporting_ccpp_v1.py',
         '--source', SOURCE, '--parent', OUTPUT], 'ccpp-train')
execute([SOURCE / 'scripts/run_false_reporting_ccpp_v1.py',
         '--source', SOURCE, '--parent', OUTPUT, '--audit-only'], 'ccpp-audit')
assert json.loads((OUTPUT / 'ccpp-comparison/audit.json').read_text())['status'] == 'verified'


## Private export and summary

The full archive includes captures and receipts. The small summary archive includes frozen inputs, heads, scores and audits but omits large captures. Neither includes credentials. A successful runtime audit must still be checked after private retrieval.


In [ ]:
for relative, result in [('audit.json', 'results.json'), ('ccpp-comparison/audit.json', 'ccpp-comparison/results.json')]:
    assert json.loads((OUTPUT / relative).read_text())['results_sha256'] == hashlib.sha256((OUTPUT / result).read_bytes()).hexdigest()
def archive_results(name, include_captures):
    destination = OUTPUT.parent / name
    descriptor, temporary = tempfile.mkstemp(dir=OUTPUT.parent, prefix='.partial-export-')
    os.close(descriptor)
    try:
        inventory = {}
        with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(OUTPUT.rglob('*')):
                if not path.is_file() or path.name.startswith('.partial'):
                    continue
                relative = path.relative_to(OUTPUT).as_posix()
                if not include_captures and relative.startswith('captures/'):
                    continue
                with path.open('rb') as stream:
                    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
                inventory[relative] = digest
                archive.write(path, relative)
            for relative in design['source_sha256']:
                archive.write(SOURCE / relative, 'source/' + relative)
            archive.writestr('archive-inventory.json', json.dumps(inventory, sort_keys=True))
        os.replace(temporary, destination)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)
    with destination.open('rb') as stream:
        checksum = hashlib.file_digest(stream, 'sha256').hexdigest()
    print(destination.name, destination.stat().st_size, 'bytes; SHA256', checksum)
    return destination
full_archive = archive_results('false-reporting-dom-ccpp-v1-complete.zip', True)
summary_archive = archive_results('false-reporting-dom-ccpp-v1-summary.zip', False)
for relative in ('results.json', 'ccpp-comparison/results.json'):
    result = json.loads((OUTPUT / relative).read_text())
    print(relative)
    for split, arms in result['splits'].items():
        print(split)
        for name, metric in arms.items():
            if isinstance(metric, dict):
                print(name, 'AUROC', round(metric['auroc'], 4),
                      'recall', round(metric['recall'], 4),
                      'false alarms', round(metric['false_positive_rate'], 4))
print('Scripted identifier-only pilot; not natural reward-hacking detection or prevention.')
shutil.copyfile(summary_archive, Path('/content') / summary_archive.name)
part_paths = []
with full_archive.open('rb') as stream:
    index = 0
    while block := stream.read(128 * 1024 * 1024):
        part = Path('/content') / (full_archive.name + '.part-' + str(index).zfill(3))
        part.write_bytes(block)
        part_paths.append(part.name)
        print('Private transfer part:', part.name, len(block), hashlib.sha256(block).hexdigest())
        index += 1
print('Download the summary and ordered parts from Files; concatenation must match the full-archive SHA256.')
